# Visualize augmentation transforms on smoke tubes

Interactive tool to inspect how each tube augmentation transform
(spatial, photometric, temporal) shapes a single smoke tube, plus the
full pipeline as configured by `params.yaml`.

Each section exposes its parameters as plain Python variables so you can
tweak and re-run the cell to see the effect.

Run after `build_model_input` has produced `data/05_model_input/{split}/`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml
from torchvision.utils import make_grid

from bbox_tube_temporal.augment import (
    PhotometricTubeTransform,
    SpatialTubeTransform,
    TemporalTubeTransform,
    build_tube_augment,
)
from bbox_tube_temporal.dataset import TubePatchDataset

## Config

In [ ]:
SPLIT = "train"
SPLIT_DIR = Path(f"../data/05_model_input/{SPLIT}")
PARAMS_PATH = Path("../params.yaml")

TUBE_INDEX = 0  # which tube to visualize (0 = first in _index.json)
NUM_VARIANTS = 6  # how many augmented copies to render per transform
MAX_FRAMES = 20
SEED = 0  # RNG seed for reproducible variants

## Helpers

`denormalize` inverts the ImageNet normalize that the aug pipeline
applies at the end (so we can display pixel values in [0, 1]).
`show_grid` assembles a list of per-variant tube tensors into one
figure with raw on top.


In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def denormalize(x: torch.Tensor) -> torch.Tensor:
    return (x * IMAGENET_STD + IMAGENET_MEAN).clamp(0.0, 1.0)


def show_grid(rows: list[torch.Tensor], row_labels: list[str], title: str) -> None:
    """Each row is a [k, 3, 224, 224] tube tensor (may have different k)."""
    max_t = max(r.shape[0] for r in rows)
    padded = [
        torch.cat(
            [r, torch.zeros(max_t - r.shape[0], *r.shape[1:], dtype=r.dtype)],
            dim=0,
        )
        if r.shape[0] < max_t
        else r
        for r in rows
    ]
    grid = make_grid(torch.cat(padded, dim=0), nrow=max_t, padding=2)
    fig, ax = plt.subplots(figsize=(max_t * 1.4, len(rows) * 1.4))
    ax.imshow(grid.permute(1, 2, 0).cpu().numpy())
    ax.set_axis_off()
    ax.set_title(title)
    # Row labels down the left edge of the grid
    row_height = grid.shape[1] / len(rows)
    for i, label in enumerate(row_labels):
        ax.text(
            -10,
            (i + 0.5) * row_height,
            label,
            ha="right",
            va="center",
            fontsize=8,
        )
    plt.show()

## Load one raw tube

Uses `TubePatchDataset(transform=None)` which returns ImageNet-normalized
patches through the legacy path. We denormalize immediately so the
rest of the notebook can feed un-normalized `[0, 1]` tubes into the
augment transforms (matches the contract those transforms expect).


In [ ]:
raw_ds = TubePatchDataset(SPLIT_DIR, max_frames=MAX_FRAMES, transform=None)
print(f"{len(raw_ds)} tubes in split")

sample = raw_ds[TUBE_INDEX]
n_valid = int(sample["mask"].sum())
seq_id = sample["sequence_id"]
label = "smoke" if float(sample["label"]) > 0.5 else "fp"
print(f"tube {TUBE_INDEX}: seq={seq_id} label={label} n_valid={n_valid}")

# Keep the raw un-normalized [0,1] patches around as a base for every
# transform test below. We reconstruct them by denormalizing the dataset
# output.
raw_patches_01 = denormalize(sample["patches"])  # [max_frames, 3, 224, 224]
raw_mask = sample["mask"].clone()


def fresh_item() -> dict:
    """Return a fresh (patches, mask) dict the transforms can mutate safely."""
    return {"patches": raw_patches_01.clone(), "mask": raw_mask.clone()}


show_grid(
    [raw_patches_01[:n_valid]],
    row_labels=["raw"],
    title=f"{seq_id} [{label}] — raw tube ({n_valid} frames)",
)

## Spatial transform

Per-tube-consistent flip, rotation, scale, translate. The same
parameters apply to every frame so motion direction is preserved.
Tweak the knobs below and re-run.


In [ ]:
SPATIAL_PARAMS = dict(
    flip_prob=0.5,
    rotation_deg=5.0,
    scale_range=(0.9, 1.1),
    translate_frac=0.05,
)

torch.manual_seed(SEED)
rows = [raw_patches_01[:n_valid]]
labels = ["raw"]
for i in range(NUM_VARIANTS):
    t = SpatialTubeTransform(**SPATIAL_PARAMS)
    out = t(fresh_item())
    rows.append(out["patches"][: int(out["mask"].sum())])
    labels.append(f"spatial #{i}")

title = f"SpatialTubeTransform({SPATIAL_PARAMS})"
show_grid(rows, row_labels=labels, title=title)

## Photometric transform

Per-tube-consistent brightness, contrast, saturation. Operates on
`[0, 1]` tensors. Hue is deliberately NOT adjusted (smoke is ~gray;
hue shifts produce out-of-distribution pink/green smoke).


In [ ]:
PHOTOMETRIC_PARAMS = dict(
    brightness_range=(0.8, 1.2),
    contrast_range=(0.8, 1.2),
    saturation_range=(0.8, 1.2),
)

torch.manual_seed(SEED)
rows = [raw_patches_01[:n_valid]]
labels = ["raw"]
for i in range(NUM_VARIANTS):
    t = PhotometricTubeTransform(**PHOTOMETRIC_PARAMS)
    out = t(fresh_item())
    rows.append(out["patches"][: int(out["mask"].sum())])
    labels.append(f"photo #{i}")

title = f"PhotometricTubeTransform({PHOTOMETRIC_PARAMS})"
show_grid(rows, row_labels=labels, title=title)

## Temporal transform

Sub-sequence sampling + stride + per-frame drop, followed by
re-compaction so valid frames still occupy `[0..k-1]` (required by
the GRU head's `pack_padded_sequence`).

Watch how tubes get shorter under aggressive drop / stride.


In [ ]:
TEMPORAL_PARAMS = dict(
    subseq_prob=0.5,
    subseq_min_len=4,
    stride_prob=0.25,
    frame_drop_prob=0.15,
    min_valid_after_drop=4,
)

torch.manual_seed(SEED)
rows = [raw_patches_01[:n_valid]]
labels = ["raw"]
for i in range(NUM_VARIANTS):
    t = TemporalTubeTransform(**TEMPORAL_PARAMS)
    out = t(fresh_item())
    rows.append(out["patches"][: int(out["mask"].sum())])
    labels.append(f"temporal #{i}")

title = f"TemporalTubeTransform({TEMPORAL_PARAMS})"
show_grid(rows, row_labels=labels, title=title)

## Full pipeline from `params.yaml`

Shows what the model actually sees during training when running
`dvc repro train_gru` (or any `train_*` stage). The pipeline is
`spatial → photometric → temporal → ImageNet normalize`; we
denormalize at the end purely for display.


In [ ]:
params = yaml.safe_load(PARAMS_PATH.read_text())
augment_cfg = params["augment"]
print(f"augment.enabled = {augment_cfg.get('enabled')}")

transform = build_tube_augment(augment_cfg, train=True)

torch.manual_seed(SEED)
rows = [raw_patches_01[:n_valid]]
labels = ["raw"]
for i in range(NUM_VARIANTS):
    # Feed raw [0,1] patches through the pipeline. The pipeline runs
    # normalize at the end, so we must denormalize the output for display.
    out = transform(fresh_item())
    k = int(out["mask"].sum())
    rows.append(denormalize(out["patches"][:k]))
    labels.append(f"full #{i}")

show_grid(rows, row_labels=labels, title="Full augment pipeline (params.yaml)")

## Custom config

Scratchpad: tweak an override dict and compare to the default pipeline
side-by-side. Useful for exploring "what if I crank up frame_drop_prob"
without editing `params.yaml`.


In [ ]:
custom_cfg = {
    "enabled": True,
    "spatial": {
        "flip_prob": 0.5,
        "rotation_deg": 10.0,  # more rotation
        "scale_range": [0.85, 1.15],  # more scale
        "translate_frac": 0.08,
    },
    "photometric": {
        "brightness_range": [0.7, 1.3],  # wider photometric
        "contrast_range": [0.7, 1.3],
        "saturation_range": [0.7, 1.3],
    },
    "temporal": {
        "subseq_prob": 0.8,
        "subseq_min_len": 4,
        "stride_prob": 0.4,
        "frame_drop_prob": 0.3,  # more drop
        "min_valid_after_drop": 4,
    },
}

custom_transform = build_tube_augment(custom_cfg, train=True)

torch.manual_seed(SEED)
rows = [raw_patches_01[:n_valid]]
labels = ["raw"]
for i in range(NUM_VARIANTS):
    out = custom_transform(fresh_item())
    k = int(out["mask"].sum())
    rows.append(denormalize(out["patches"][:k]))
    labels.append(f"custom #{i}")

show_grid(rows, row_labels=labels, title="Custom augment pipeline")